# Improve an agent's harness

<a id="harness-evolution"></a>

Our help assistant answers “reset password” but misses “please reset password.”
We'll improve the instruction it follows, then test the answers it produces.
A **harness** is the instructions and settings around an agent; here it holds
one matching instruction.

**Change the instruction → run the same assistant → test its answers → keep the best measured settings.**

This five-cell lesson is self-contained. The assistant and proposer are
handwritten Python simulations, with no model, API key, or earlier notebook
required. The assistant recognizes exactly two instructions; it does not
understand arbitrary prompts.



## 1. Install

Use a fresh notebook environment running **Python 3.12 or newer**.
Install directly from the published documentation:

In [ ]:
%pip install https://sentient-xyz.github.io/meta-evolve-docs/downloads/meta-evolve.zip

If you already imported Meta-Evolve, restart the kernel after installing.
Then run the remaining cells in order.

**Archived or offline docs:** use the ZIP included with that build. Put
`meta-evolve.zip` in the notebook's working folder (`%pwd` shows it; hosted
notebooks let you upload files), then run `%pip install ./meta-evolve.zip`
instead. Installing from source may still download build tools.

<a id="2-declare-what-can-change"></a>
<a id="3-connect-the-configuration-to-an-assistant"></a>

## 2. Start with one instruction and three questions

`HARNESS_SEED` is the starting configuration. Its `planner` instruction requires
an exact question match. The assistant's `HELP_NOTES` stay fixed: it can answer
password and invoice questions if its matching rule finds the topic.

`answer` receives a question and the configuration. `HELP_CASES` contains the
questions and expected answers used to measure it. These checks stay outside
the configuration being revised.

The [fixed-evaluation notebook](https://sentient-xyz.github.io/meta-evolve-docs/concepts/authority/) extends this story
with a typed mutation boundary: a permitted mistake is graded, while a change
to a protected declaration is refused before evaluation. That additional
validation does not apply to the plain dictionary used here.

In [ ]:
import meta_evolve as meta

HARNESS_SEED = {"planner": "Match exact questions."}
HELP_NOTES = (
    ("reset password", "Open Settings > Password."),
    ("download invoice", "Open Billing > Invoices."),
)


def answer(question, configuration):
    instruction = configuration["planner"]
    if instruction not in ("Match exact questions.", "Match topic words."):
        raise ValueError("Unsupported matching instruction")
    question = question.lower()
    for topic, reply in HELP_NOTES:
        if instruction == "Match exact questions.":
            matches = question == topic
        else:
            matches = set(topic.split()) <= set(question.split())
        if matches:
            return reply
    return "I don't have a matching help note."


# These expected answers stay fixed while the instruction changes.
HELP_CASES = (
    ("reset password", "Open Settings > Password."),
    ("please reset password", "Open Settings > Password."),
    ("download invoice", "Open Billing > Invoices."),
)

<a id="4-grade-answers-independently"></a>

## 3. Test answers and retain the checks

`evaluate_harness` asks the assistant each question and compares the actual
answer with the expected answer. The score is the fraction of checks that pass.
The retained **evidence** contains each question, both answers, and the result.

In [ ]:
def evaluate_harness(configuration):
    checks = []
    for question, expected in HELP_CASES:
        actual = answer(question, configuration)
        checks.append({"question": question, "expected": expected,
                       "actual": actual, "passed": actual == expected})
    return meta.EvaluationResult(
        metrics={"score": sum(check["passed"] for check in checks) / len(checks)},
        evidence=(meta.EvidenceDraft(kind="help-checks", data={"checks": checks}),),
    )

A missing help note is an ordinary answer that can fail a check. An unsupported
instruction raises an execution error; Meta-Evolve records an evaluation failure
with no score.

<a id="5-propose-one-change-at-a-time"></a>

## 4. Propose a broader matching instruction

Our handwritten proposer changes `planner` to `Match topic words.`. With this
instruction, the assistant accepts extra words around a topic, so “please reset
password” can match “reset password.” It returns a new dictionary without changing
the parent. The notes, questions, and expected answers stay the same.

In [ ]:
def propose_harness(parent):
    # Change the instruction, preserving the previous configuration.
    return {**parent, "planner": "Match topic words."}

<a id="6-run-and-inspect-development-search"></a>

## 5. Run, inspect, and use the settings

`improve` evaluates the starting settings, tries the proposed revision, and
keeps the version with the better score. `trials=1` stops this run after one
proposal attempt. Testing the starting settings and the revision uses two
evaluations; each evaluation checks all three questions.

The output below comes from the recorded checks. The starting version can
still win when a proposed change does not improve its score.

In [ ]:
harness_result = meta.improve(
    seed=HARNESS_SEED,
    proposer=propose_harness,
    evaluator=evaluate_harness,
    trials=1,
)
for label, version in (("Starting", harness_result.trials()[0]),
                       ("Selected", harness_result.best_trial())):
    checks = version.evidence[0].data["checks"]
    print(f"{label}: {sum(check['passed'] for check in checks)}/{len(checks)} checks")
    for check in checks:
        print(check["question"], "->", check["actual"])

selected_harness = harness_result.best().value
print("Selected configuration:", dict(selected_harness))
print("Attempts:", harness_result.usage().trials)
print("Evaluations:", harness_result.usage().evaluations)
print("Reuse:", answer("please download my invoice", selected_harness))
# Output:
# Starting: 2/3 checks
# reset password -> Open Settings > Password.
# please reset password -> I don't have a matching help note.
# download invoice -> Open Billing > Invoices.
# Selected: 3/3 checks
# reset password -> Open Settings > Password.
# please reset password -> Open Settings > Password.
# download invoice -> Open Billing > Invoices.
# Selected configuration: {'planner': 'Match topic words.'}
# Attempts: 1
# Evaluations: 2
# Reuse: Open Billing > Invoices.

Only the matching instruction changed. The polite password answer now passes,
while the two previously correct answers still pass: **2/3 → 3/3**.
`selected_harness` holds the settings you can pass back to the assistant.
`HARNESS_SEED` remains unchanged.

The last call uses the selected settings for another question. This is a reuse
demonstration, not a held-out quality measurement. It happens outside the run's
two accounted evaluations. These results describe our three questions and
handwritten simulation; they do not establish real-model or general quality.

## Change and predict

Change `trials=1` to `trials=0` and rerun the last cell. The starting settings
remain selected at **2/3**. “Please reset password” still has no matching note,
and the reuse question also fails to match. There are zero attempts and one
evaluation.

Restore `trials=1`, then add `("recover my credentials", "Open Settings > Password.")`
to `HELP_CASES` and rerun cells 2–5. Neither matching instruction understands
that phrasing: the selected settings pass **3/4** checks. Changing an expected
answer also changes the measurement; the evaluator checks returned answers,
not the name of the instruction.

To adapt the example, replace `answer` with your agent, `HELP_CASES` with your
fixed checks, and `propose_harness` with your revision function.
[Use your model SDK](https://sentient-xyz.github.io/meta-evolve-docs/guides/providers/) shows how to connect actual model calls.

## Go further

A plain dictionary is enough for this instruction experiment. The separate
[Meta-Harness repair study](https://sentient-xyz.github.io/meta-evolve-docs/research/meta-harness/#the-concrete-problem)
changes a worker's instructions, runs repository repairs, and independently
tests the produced code. Its
[typed harness declaration](https://sentient-xyz.github.io/meta-evolve-docs/research/meta-harness/#declare-allowed-changes)
also validates which components and interfaces may change. Those structured
mutation guarantees do not apply to the plain dictionary used here.

<a id="7-check-fresh-questions-after-selection"></a>

### Check fresh work after selection

Search results measure the cases used during search. The repair study adds
[private selection and a controlled held-out comparison](https://sentient-xyz.github.io/meta-evolve-docs/research/meta-harness/#check-fresh-repairs)
with fixed evaluation rules and budgets. Follow that path when you need an
independent check before adopting selected settings.

<a id="8-export-and-reuse-the-selected-harness"></a>

### Export and reuse a structured harness

Here, reuse means passing `selected_harness` to `answer`. The repair study
demonstrates [eligible export of a typed harness](https://sentient-xyz.github.io/meta-evolve-docs/research/meta-harness/#export-and-adopt)
and leaves adoption to the caller. Its specialized comparison and export APIs
operate on structured harnesses. To retain this simple run's history, follow
[Save and reopen a run](https://sentient-xyz.github.io/meta-evolve-docs/learn/06-persist-run/); saving local history does
not make its callables resumable.

[Improve a search policy](https://sentient-xyz.github.io/meta-evolve-docs/guides/policy-evolution/) · [Choose an example](https://sentient-xyz.github.io/meta-evolve-docs/examples/)